# 13 ? Business Interpretation and Recommendations

**Production note:** This notebook reads the validated production outputs from `scripts/` and `data/outputs/`. It does not re-fit models or re-run model selection. All metrics and forecast values shown by code cells are loaded from the existing CSV outputs, so the notebook does not hand-enter model results.

Scope:

- Restate the selected model result for each target region.
- Show training history, 2025 validation actuals, selected 2025 prediction, and selected 2026-2027 forecast for each target.
- Interpret model limitations, feature meaning, and business implications for Repsol.
- Identify internal Repsol data that would make the modeling stronger.


## 0. Setup ? Load Validated Production Outputs

In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import subprocess
import sys
from IPython.display import display, Markdown

cwd = Path().resolve()
REPO_ROOT = cwd if (cwd / 'data').exists() and (cwd / 'notebooks').exists() else cwd.parent
DATA_FEATURES = REPO_ROOT / 'data' / 'features'
DATA_OUTPUTS = REPO_ROOT / 'data' / 'outputs'
FIGS = REPO_ROOT / 'reports' / 'figures'
FIGS.mkdir(parents=True, exist_ok=True)

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
TARGET_COLORS = {
    'Nacional': '#FF6B35',
    'Madrid': '#004E89',
    'Cataluña': '#1A936F',
    'Andalucía': '#C84B31',
    'Valencia': '#8E44AD',
}
FINAL_MODEL_POLICY = {
    'Nacional': 'SARIMA',
    'Madrid': 'Logistic',
    'Cataluña': 'SARIMA',
    'Andalucía': 'Logistic',
    'Valencia': 'Gompertz',
}

required_files = {
    'features': DATA_FEATURES / 'features_modelo_completo.csv',
    'final_metrics': DATA_OUTPUTS / 'metricas_final_selected.csv',
    'all_metrics': DATA_OUTPUTS / 'metricas_models.csv',
    'predictions': DATA_OUTPUTS / 'predicciones_test_2025.csv',
    'forecasts': DATA_OUTPUTS / 'forecast_24m_sarima_rf_xgb.csv',
    'acceptance': DATA_OUTPUTS / 'phase2_model_acceptance.csv',
    'pooling': DATA_OUTPUTS / 'phase2_pooling_decision.csv',
    'sarima_grid': DATA_OUTPUTS / 'sarima_grid_search_results.csv',
    'sarima_acceptance': DATA_OUTPUTS / 'sarima_order_acceptance.csv',
}
missing = [str(path.relative_to(REPO_ROOT)) for path in required_files.values() if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing required production output files: {missing}')

features = pd.read_csv(required_files['features'])
final_metrics = pd.read_csv(required_files['final_metrics'])
all_metrics = pd.read_csv(required_files['all_metrics'])
predictions = pd.read_csv(required_files['predictions'])
forecasts = pd.read_csv(required_files['forecasts'])
acceptance = pd.read_csv(required_files['acceptance'])
pooling = pd.read_csv(required_files['pooling'])
sarima_grid = pd.read_csv(required_files['sarima_grid'])
sarima_acceptance = pd.read_csv(required_files['sarima_acceptance'])

for df in [features, predictions, forecasts]:
    df['Fecha_Date'] = pd.to_datetime(df['Fecha'])

selected_models = dict(zip(final_metrics['Target'], final_metrics['Model']))
if selected_models != FINAL_MODEL_POLICY:
    print('Detected stale notebook-generated model outputs. Rebuilding production modeling outputs...')
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '05_modeling_with_cnmc.py')], check=True, cwd=REPO_ROOT)
    final_metrics = pd.read_csv(required_files['final_metrics'])
    all_metrics = pd.read_csv(required_files['all_metrics'])
    predictions = pd.read_csv(required_files['predictions'])
    forecasts = pd.read_csv(required_files['forecasts'])
    acceptance = pd.read_csv(required_files['acceptance'])
    pooling = pd.read_csv(required_files['pooling'])
    sarima_grid = pd.read_csv(required_files['sarima_grid'])
    sarima_acceptance = pd.read_csv(required_files['sarima_acceptance'])
    for df in [predictions, forecasts]:
        df['Fecha_Date'] = pd.to_datetime(df['Fecha'])
    selected_models = dict(zip(final_metrics['Target'], final_metrics['Model']))
    if selected_models != FINAL_MODEL_POLICY:
        raise ValueError(f'Final selected model policy mismatch after rebuild: {selected_models}')

print('Loaded production outputs from:', REPO_ROOT)
print('Final selected models:', selected_models)


## 1. What the Models Can and Cannot Tell Us

The current project estimates **total market biodiesel demand** for the national total and four selected regions. It does **not** estimate Repsol's own sales, station-level volumes, margins, or logistics needs.

What the current models can support:

- Directional 2026-2027 market-demand planning by target region.
- Comparison of candidate forecasting approaches on the validated 2025 acceptance period.
- Identification of where the model is relatively more or less reliable.
- Discussion of which additional Repsol internal data would make the forecast operationally stronger.

What the current models cannot prove alone:

- Repsol market share by region.
- Station-level product demand.
- Inventory or logistics requirements.
- Price elasticity specific to Repsol customers.
- Financial impact or ROI of commercial actions.


### 1.1 Exact Selected Results by Region

The table below is loaded directly from `data/outputs/metricas_final_selected.csv`. These are the final selected 2025 validation metrics.

In [ ]:
display(final_metrics.sort_values('Target').reset_index(drop=True))

### 1.2 Exact Selection and Acceptance Decisions

The table below is loaded directly from `data/outputs/phase2_model_acceptance.csv`. It shows the Phase 1 baseline, Phase 2 proposed model, final selection source, and final decision.

In [ ]:
display(acceptance.sort_values('Target').reset_index(drop=True))

### 1.3 SARIMA Grid-Search Robustness Check

SARIMA parameter tuning is now evaluated before final selection using a constrained grid inside the 2023-2024 training period only. The grid winner is then passed through a 2025 no-regression acceptance check versus the original default SARIMA order. This keeps the grid search informative without allowing a training-CV winner to silently weaken the accepted production forecast.


In [ ]:
display(Markdown('#### Training-only SARIMA grid winners'))
display(
    sarima_grid[sarima_grid['Selected']]
    [['Target', 'p', 'd', 'q', 'P', 'D', 'Q', 'm', 'WalkForward_MAPE', 'Successful_Folds']]
    .sort_values('Target')
    .reset_index(drop=True)
)

display(Markdown('#### SARIMA production-order acceptance'))
display(
    sarima_acceptance[
        [
            'Target', 'Default_Order', 'Default_Seasonal_Order',
            'Grid_Selected_Order', 'Grid_Selected_Seasonal_Order',
            'Default_2025_MAPE', 'Grid_Selected_2025_MAPE',
            'Production_Order', 'Production_Seasonal_Order', 'Decision'
        ]
    ]
    .sort_values('Target')
    .reset_index(drop=True)
)


### 1.3 Candidate Model Metrics by Region

The following tables sort every tested candidate by validation MAPE within each target. These tables are loaded directly from `data/outputs/metricas_models.csv`.

In [ ]:
for target in TARGETS:
    display(Markdown(f'#### {target}'))
    cols = ['Target', 'Model', 'MAE', 'RMSE', 'MAPE', 'R2']
    display(
        all_metrics.loc[all_metrics['Target'].eq(target), cols]
        .sort_values(['MAPE', 'Model'])
        .reset_index(drop=True)
    )


### 1.4 Regional Training, Validation, and Forecast Plots

Each plot is generated from the production feature, prediction, and forecast CSVs:

- Training actuals: 2023-01 to 2024-12.
- 2025 validation actuals.
- 2025 prediction from the final selected model.
- 2026-2027 forecast from the final selected model.

The generated image files are saved under `reports/figures/` so they can be reused in slides or the final report.

In [ ]:
def safe_target_name(target: str) -> str:
    replacements = {
        'Nacional': 'national',
        'Madrid': 'madrid',
        'Cataluña': 'cataluna',
        'Andalucía': 'andalucia',
        'Valencia': 'valencia',
    }
    return replacements[target]

figure_rows = []
for target in TARGETS:
    model = selected_models[target]
    hist = features.loc[features['Target'].eq(target)].sort_values('Fecha_Date')
    train = hist.loc[hist['Fecha'].lt('2025-01')]
    valid = hist.loc[hist['Fecha'].ge('2025-01')]
    pred = predictions.loc[predictions['Target'].eq(target) & predictions['Model'].eq(model)].sort_values('Fecha_Date')
    fc = forecasts.loc[forecasts['Target'].eq(target) & forecasts['Model'].eq(model)].sort_values('Fecha_Date')
    metric = final_metrics.loc[final_metrics['Target'].eq(target)].iloc[0]

    if pred.empty:
        raise ValueError(f'Missing 2025 prediction rows for {target} / {model}')
    if fc.empty:
        raise ValueError(f'Missing 2026-2027 forecast rows for {target} / {model}')

    fig, ax = plt.subplots(figsize=(12, 5))
    ax.plot(train['Fecha_Date'], train['Consumo_Tm'], color='#333333', linewidth=2.2, label='Training actuals 2023-2024')
    ax.plot(valid['Fecha_Date'], valid['Consumo_Tm'], color='#777777', linewidth=2.2, label='2025 validation actuals')
    ax.plot(pred['Fecha_Date'], pred['Pred'], color='#D95F02', linewidth=2.2, linestyle='--', label=f'2025 selected prediction ({model})')
    ax.plot(fc['Fecha_Date'], fc['Forecast'], color=TARGET_COLORS[target], linewidth=2.4, label=f'2026-2027 selected forecast ({model})')
    ax.axvline(pd.Timestamp('2025-01-01'), color='#999999', linestyle=':', linewidth=1.3)
    ax.axvline(pd.Timestamp('2026-01-01'), color='#999999', linestyle=':', linewidth=1.3)
    ax.set_title(f'{target} ? selected model: {model} | 2025 MAPE: {metric.MAPE:.1f}% | R2: {metric.R2:.3f}', fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('Biodiesel demand (metric tonnes)')
    ax.grid(True, alpha=0.25)
    ax.legend(loc='upper left', fontsize=8)
    fig.tight_layout()

    out = FIGS / f'13_business_{safe_target_name(target)}_train_validation_forecast.png'
    fig.savefig(out, dpi=150, bbox_inches='tight')
    plt.close(fig)
    figure_rows.append({'Target': target, 'Model': model, 'Figure': str(out.relative_to(REPO_ROOT))})

figure_manifest = pd.DataFrame(figure_rows)
display(figure_manifest)


#### Nacional

![Nacional training, validation, and forecast](../reports/figures/13_business_national_train_validation_forecast.png)


#### Madrid

![Madrid training, validation, and forecast](../reports/figures/13_business_madrid_train_validation_forecast.png)


#### Cataluña

![Cataluña training, validation, and forecast](../reports/figures/13_business_cataluna_train_validation_forecast.png)


#### Andalucía

![Andalucía training, validation, and forecast](../reports/figures/13_business_andalucia_train_validation_forecast.png)


#### Valencia

![Valencia training, validation, and forecast](../reports/figures/13_business_valencia_train_validation_forecast.png)


### 1.5 Existing Summary Figures from Earlier Notebooks

These are the existing production figures created by the previous notebooks / script outputs. They are included here for continuity with the modeling work already completed.

![Train/test split](../reports/figures/10_train_test_split.png)

![Actual vs predicted validation results](../reports/figures/11_predictions_vs_actuals.png)

![Selected 24-month forecast](../reports/figures/11_forecast_24m.png)


## 2. Model-by-Model Limitations

| Model family | Where it appears in this project | Main limitations for this project |
|---|---|---|
| SARIMA | Final selected model for Nacional and Cataluña; candidate for all targets | Uses the target history and seasonality, but does not directly use macro, diesel-market, mandate, price, or Repsol-specific business variables. With only 36 months of history, seasonal structure and trend persistence are fragile. |
| Logistic growth curve | Final selected model for Madrid; candidate for regional targets | Imposes a smooth saturation curve plus seasonal correction. This is useful for adoption-like growth, but it can be wrong if policy, supply, competition, or customer behavior changes abruptly. |
| Gompertz growth curve | Final selected model for Valencia; candidate for regional targets | Also imposes a saturating adoption curve plus seasonal correction. The asymptote is estimated from a short history, so the long-run level should be treated cautiously. |
| Ridge | Candidate model only | Sensitive to collinearity among lag, trend, and rolling features. It can extrapolate unstable trends when used recursively over many months. |
| Random Forest | Candidate model and pooled sensitivity model | Can capture nonlinear patterns, but tends to interpolate within learned ranges and can flatten forecasts. With a very small monthly dataset, feature importance and validation performance can be unstable. |
| XGBoost | Candidate model and pooled sensitivity model | Flexible but data-hungry. With limited history, it can overfit candidate patterns or fail to extrapolate structural adoption behavior. |
| Diesel Share | Candidate model only | Depends on diesel-market proxy assumptions and future diesel-market extrapolation. It is useful as a business-logic benchmark, not as a final selected model in this project. |
| Pooled regional ML | Sensitivity experiment only | Increases effective training rows by stacking regions, but may blur regional dynamics. The final project policy is non-pooled, so pooled models are not used in selected production forecasts. |


## 3. Why Model Performance Differs by Region

The exact candidate rankings are shown in Section 1.3. The interpretation below explains why different model families are plausible for different targets without replacing the metric tables.

| Target | Final selected model | Interpretation |
|---|---|---|
| Nacional | SARIMA | The national series aggregates regional variation, so a target-history model can be more defensible than regional curve or ML models. The national result is still a planning forecast, not a high-precision commitment. |
| Madrid | Logistic | Madrid uses a saturating growth curve in the final selection. This suggests the selected model handled the observed regional adoption pattern better than the Phase 1 baseline under the accepted validation process. |
| Cataluña | SARIMA | Cataluña is the key policy tradeoff: pooled Random Forest appears as a sensitivity result, but the final delivery policy is non-pooled, so SARIMA is used as the selected production model. |
| Andalucía | Logistic | The final selection keeps the model that did not worsen validation performance relative to the Phase 1 baseline. The selected curve model should still be interpreted cautiously because the region has limited monthly history. |
| Valencia | Gompertz | Valencia keeps a saturating curve model. The business interpretation is that growth is treated as adoption-like rather than indefinitely exponential, but the estimated ceiling remains uncertain. |


## 4. Feature and Driver Interpretation

Feature importance is only directly meaningful for models that actually use feature columns. The final selected model set contains SARIMA and growth-curve models, so standard tree or coefficient feature importance does **not** apply uniformly across the final selected models.

| Final model type | What drives the model in this project | What should not be overclaimed |
|---|---|---|
| SARIMA | Past target values, trend persistence, and seasonal structure in the target series. | It does not estimate causal effects of macro variables, mandates, diesel market size, or Repsol commercial levers. |
| Logistic / Gompertz | Time trend, saturation shape, and a small seasonal correction based on monthly sine/cosine terms. | It does not identify causal business drivers; it represents an adoption-curve shape. |
| ML candidates | Lagged target, rolling target features, lagged macro indicators, lagged CNMC diesel-market features, and mandate variables. | Candidate-model feature importance is diagnostic only; it should not be presented as final selected-model causality. |
| Pooled ML sensitivity | Same broad feature families as the regional ML candidates, with region indicators. | Useful for sensitivity comparison, but not used in the final non-pooled production forecast. |

The existing feature-importance figure from notebook 09 is included below as a candidate-model diagnostic, not as proof of causal drivers for every final selected model.

![Feature importance diagnostic](../reports/figures/10_feature_importance.png)


## 5. What Non-Public Internal Repsol Data Would Improve the Forecasts

The current model predicts market demand. To convert this into stronger Repsol-specific planning, the most valuable additions would be internal operational and commercial data.

| Internal data category | Examples | Why it would improve the model |
|---|---|---|
| Repsol sales volumes | Monthly or weekly sales by station, product, region, and customer segment. | Separates total market demand from Repsol-specific demand and market share. |
| Station-level demand | Station coordinates, catchment area, local traffic, product availability, and station format. | Allows local demand modeling instead of only CCAA-level market forecasting. |
| Repsol pricing and promotions | Pump prices, discounts, loyalty offers, B2B contract prices, campaign dates. | Enables price-response and promotion-response modeling. |
| Product availability | Biodiesel/HVO availability by station and date, rollout dates, stockout records. | Distinguishes true demand from constrained sales caused by supply or availability. |
| Inventory and logistics | Tank capacity, replenishment dates, delivery lead times, stock levels, distribution constraints. | Connects demand forecasts to operational planning and risk of shortages. |
| Customer behavior | Fleet accounts, loyalty-card behavior, repeat customers, B2B vs retail split. | Improves segmentation and identifies demand sources with different behavior. |
| Competitive context | Nearby competitor stations, competitor prices, local diesel alternatives. | Helps explain regional or station-level substitution patterns. |
| Policy and compliance data | Internal compliance targets, blending strategy, contract obligations. | Connects market demand forecasts to Repsol's regulatory and strategic constraints. |


## 6. Business Recommendations for Repsol

These recommendations are intentionally framed as planning guidance, because the current project does not include Repsol internal sales or margin data.

1. Use the current forecast as a **market-demand scenario**, not as a direct Repsol sales forecast.
2. Treat regional forecasts as **directional signals** and monitor actuals closely, especially where validation metrics are weak.
3. Maintain the non-pooled forecast as the official deliverable, but keep pooled regional ML as a sensitivity check for future model development.
4. Build a Repsol-specific demand layer on top of the market forecast using internal sales, pricing, station, availability, and customer data.
5. Use the model outputs to support regional scenario planning, not single-point operational commitments.
6. Re-run the pipeline when newer actual demand data becomes available and track forecast drift by region.
7. For business decisions such as inventory, pricing, and logistics, require internal data validation before acting on model outputs.


## 7. Priority Data Roadmap

| Priority | Data / modeling improvement | Purpose |
|---:|---|---|
| 1 | Add Repsol sales by station, product, customer type, and month. | Convert market-demand forecasting into Repsol-demand forecasting. |
| 2 | Add product availability, inventory, and stockout history. | Separate demand from supply constraints. |
| 3 | Add Repsol pricing, discounts, promotions, and B2B contract terms. | Estimate commercial levers and price sensitivity. |
| 4 | Add customer/fleet segmentation and loyalty behavior. | Improve regional and station-level demand segmentation. |
| 5 | Add competitor/local market context around stations. | Explain local substitution and market-share differences. |
| 6 | Extend the actual demand history when new 2026 data becomes available. | Strengthen backtesting and reduce reliance on a single 2025 validation year. |
| 7 | Develop a two-layer model: market demand first, Repsol share second. | Keep the current capstone model useful while making it operational for Repsol. |


## Final Caution

The most important business message is that the current project is a solid market-demand forecasting framework, but not yet a full Repsol operational forecast. The next step is not simply a more complex algorithm; it is adding Repsol internal data so the model can distinguish market growth from Repsol-specific sales, availability, pricing, customer behavior, and logistics constraints.
